# Tidy data: one observation per row, one variable per column

_Data Wrangling_

**Reshape messy tables into the long, tidy form that grouping, plotting, and modeling all assume.**

Hadley Wickham's tidy data is one simple rule with three parts. In a tidy table:

       
         
- every row is one observation (one measured thing at one time);
         
- every column is one variable (one kind of measurement);
         
- every cell is one value.
       

---

This notebook is a practice scaffold for the **Tidy data: one observation per row, one variable per column** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — pandas

In [ ]:
import pandas as pd

# --- An untidy WIDE table: columns are YEARS (values, not variable names) ---
# GDP-per-person, in thousands (illustrative numbers).
wide = pd.DataFrame({
    "country": ["Brazil", "India", "Nigeria", "Vietnam"],
    "1999":    [3.10, 0.45, 0.50, 0.36],
    "2009":    [8.60, 1.10, 1.10, 1.20],
    "2019":    [8.80, 2.10, 2.20, 3.40],
})
print(wide.shape)          # -> (4, 4): 4 rows, 4 columns (3 of them are years)

# === melt: WIDE -> LONG (tidy) =========================================
# Keep 'country' as an id; turn the year COLUMN NAMES into a 'year' column
# and the cell VALUES into a 'gdp_pc' column.
long = wide.melt(id_vars="country", var_name="year", value_name="gdp_pc")
print(long.shape)          # -> (12, 3): one observation per row now
#    country  year  gdp_pc
# 0   Brazil  1999    3.10
# 1    India  1999    0.45
# ...        (12 rows total)

# Tidy data makes downstream work trivial: this groupby is a one-liner now,
# but is impossible while 'year' is smeared across column headers.
print(long.groupby("year")["gdp_pc"].mean())

# === pivot_table: LONG -> WIDE (for humans) ============================
# Spread 'year' back into columns. pivot_table (not plain pivot) takes an
# aggfunc, so it is safe even if a (country, year) pair repeats.
back = long.pivot_table(index="country", columns="year",
                        values="gdp_pc", aggfunc="mean")
back = back.reset_index()  # lift 'country' out of the index so it stays a column
print(back)

# stack / unstack do the same reshape over the INDEX instead of columns:
# long.set_index(["country", "year"])["gdp_pc"].unstack("year")  == back's body

# === splitting / combining: many values in one cell ===================
messy = pd.DataFrame({
    "user":  ["u1", "u2"],
    "group": ["male_25-34", "female_18-24"],   # TWO variables in one column
    "tags":  ["sports,news", "music"],          # a LIST packed into one cell
})

# split a combined column into two real columns
messy[["sex", "age"]] = messy["group"].str.split("_", expand=True)

# explode a delimited cell into one row per value
tidy = (messy.drop(columns="group")
             .assign(tags=messy["tags"].str.split(","))
             .explode("tags")
             .reset_index(drop=True))
print(tidy)
#   user         tags     sex     age
# u1          sports    male   25-34
# u1            news    male   25-34
# u2           music  female   18-24

## Visualize the data & results

_Melting a wide table (years as columns) into tidy long form: how do the row and column counts change?_

In [ ]:
import pandas as pd

wide = pd.DataFrame({
    "country": ["Brazil", "India", "Nigeria", "Vietnam"],
    "1999":    [3.10, 0.45, 0.50, 0.36],
    "2009":    [8.60, 1.10, 1.10, 1.20],
    "2019":    [8.80, 2.10, 2.20, 3.40],
})
long = wide.melt(id_vars="country", var_name="year", value_name="gdp_pc")

print("wide rows, cols:", wide.shape)   # -> (4, 4)
print("long rows, cols:", long.shape)   # -> (12, 3)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A sales export has columns store, Jan, Feb, Mar with revenue in each month cell. You try df.groupby("month")["revenue"].sum() and get a KeyError: 'month'. Why, and how do you fix the shape?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice the month columns Jan/Feb/Mar are values of a hidden month variable, not three separate variables. — _This is the "column headers are values" mess: the table is wide and untidy._
- There is no month column and no revenue column for groupby to use, so the key lookup fails. — _groupby needs the grouping variable to actually be a column._
- Melt: df.melt(id_vars="store", var_name="month", value_name="revenue"). — _This stacks the three month columns into a month column and the cells into a revenue column &mdash; now both keys exist._

**Answer:** The table is wide: Jan/Feb/Mar are values of month, so neither a month nor a revenue column exists for groupby &mdash; hence the KeyError. Reshape to long with df.melt(id_vars="store", var_name="month", value_name="revenue"), then groupby("month")["revenue"].sum() works.

</details>

**Problem 2.** A column group contains strings like "male_25-34" and "female_18-24", and a column tags contains comma-strings like "sports,news". What two operations make this tidy?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize "male_25-34" packs two variables (sex and age band) into one cell, and "sports,news" packs a list into one cell. — _Tidy data requires one value per cell and one variable per column._
- Split the combined column: df[["sex","age"]] = df["group"].str.split("_", expand=True). — _This separates the two facts into two proper columns._
- Explode the list column: df.assign(tags=df["tags"].str.split(",")).explode("tags"). — _explode gives each list item its own row, so each cell holds one value._

**Answer:** Use str.split to break "male_25-34" into separate sex and age columns (str.split("_", expand=True)), and explode to turn "sports,news" into one row per tag (split on "," then explode). Now every cell holds a single value.

</details>

**Problem 3.** You run long.pivot(index="country", columns="year", values="gdp") and get ValueError: Index contains duplicate entries, cannot reshape. What does this mean and what should you use instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the error: more than one row maps to the same (country, year) cell. — _Plain pivot requires every row&times;column pair to be unique; it has no way to combine two values into one cell._
- Decide how duplicates should be combined &mdash; e.g. averaged or summed. — _You must tell pandas the aggregation; it will not guess._
- Switch to long.pivot_table(index="country", columns="year", values="gdp", aggfunc="mean"). — _pivot_table accepts an aggfunc and collapses the duplicates on purpose._

**Answer:** It means several rows share the same (country, year) key, and plain pivot cannot put two values in one cell. Use pivot_table with an explicit aggfunc (e.g. "mean" or "sum") to aggregate the duplicates into a single value per cell.

</details>